## 1. Problem Understanding

This notebook implements a small reasoning-focused GenAI prototype for regulatory compliance checking.

The task has three main requirements:

1. Extract structured constraints from a municipal construction notice  
   → Transform free text into a well-defined JSON schema of constraints

2. Evaluate these constraints against a simplified ruleset  
   → Identify compliance, partial compliance, or violations with explicit reasoning

3. Produce a machine-readable compliance assessment  
   → Output should be suitable for downstream automation or agent-based workflows

In this solution, the LLM is used for extracting structured constraints from the German municipal letter.  
The compliance evaluation itself is implemented with deterministic Python logic, because the ruleset is explicit and the results should be explainable. 

The overall workflow is:

Municipal Letter  
→ LLM-based constraint extraction  
→ Pydantic schema validation  
→ Python rule evaluation  
→ Machine-readable compliance assessment JSON

## 2. Load Input Documents
In this section, we load the municipal letter and ruleset from DOCX files and convert them into plain text.

In [7]:
from docx import Document
from pathlib import Path

In [8]:
def read_docx(file_path: str) -> str:
    
    doc = Document(file_path)

    paragraphs = [
        paragraph.text.strip()
        for paragraph in doc.paragraphs
        if paragraph.text.strip()
    ]

    return "\n".join(paragraphs)

In [ ]:
municipal_letter_path = "PATH TO \Municipal_Letter.docx"

municipal_text = read_docx(
    municipal_letter_path
)

print(municipal_text)

Hiermit genehmigen wir den Ausbau der Glasfasertrasse im Bereich Musterstraße.
Dabei sind folgende Auflagen einzuhalten:
Die maximale Grabentiefe darf 80 cm nicht unterschreiten.
Die Bauarbeiten sind ausschließlich in der Zeit von 7:00 bis 17:00 Uhr durchzuführen.
Offene Gräben sind über Nacht ordnungsgemäß abzudecken oder zu sichern.
Erdarbeiten im unmittelbaren Bereich von Bäumen sind nicht zulässig.
Ein wöchentlicher Fortschrittsbericht ist der zuständigen Behörde vorzulegen.


In [ ]:
ruleset_path = "PATH TO \Ruleset.docx"

ruleset_text = read_docx(
    ruleset_path
)

print(ruleset_text)

R1: Mindestgrabentiefe beträgt mindestens 60 cm
R2: Bauarbeiten sind zwischen 6:00 und 20:00 Uhr zulässig
R3: Offene Gräben müssen entweder gesichert oder abgedeckt werden
R4: Arbeiten innerhalb von 2 m um geschützte Bäume sind verboten
R5: Wöchentliche Fortschrittsberichte sind zulässig, aber nicht verpflichtend


In [11]:
assert municipal_text
assert ruleset_text

print("Documents loaded successfully.")

Documents loaded successfully.


## 3. Schema Definition

Structured schemas are defined for the extracted constraints and the final compliance assessment.

These schemas ensure consistent validation, rule evaluation, and machine-readable outputs across the workflow.

In [14]:
from pydantic import BaseModel, Field
from typing import Optional, Literal, List


class TimeWindow(BaseModel):
    start_time: Optional[str] = None
    end_time: Optional[str] = None


class Constraint(BaseModel):
    constraint_id: str
    source_text: str
    constraint_type: str
    parameter: str
    operator: str
    value: Optional[float | str] = None
    unit: Optional[str] = None
    time_window: Optional[TimeWindow] = None
    condition: Optional[str] = None
    obligation: Literal["required", "forbidden", "allowed", "optional"]
    confidence: float = Field(ge=0.0, le=1.0)


class ExtractedConstraints(BaseModel):
    document_name: str
    constraints: List[Constraint]

In [16]:
class ComplianceResult(BaseModel):
    rule_id: str
    rule_text: str
    related_constraint_id: Optional[str]
    status: Literal[
        "compliant",
        "violation",
        "partial_compliance",
        "not_applicable",
        "needs_review"
    ]
    evidence_from_letter: Optional[str]
    reasoning: str
    recommendation: Optional[str] = None


class ComplianceAssessment(BaseModel):
    document_name: str
    overall_status: Literal[
        "compliant",
        "violation",
        "partial_compliance",
        "needs_review"
    ]
    results: List[ComplianceResult]

## 4. LLM API Setup
In this section, we configure the connection to the provided LLM API.

The API credentials are loaded from environment variables instead of being hardcoded in the notebook.
This avoids exposing secrets in GitLab.

In [81]:
import os
import requests
from dotenv import load_dotenv

load_dotenv("PATH TO .env")

False

In [82]:
CLIENT_ID = os.getenv("CLIENT_ID")
CLIENT_SECRET = os.getenv("CLIENT_SECRET")
TOKEN_ISSUER_URL = os.getenv("TOKEN_ISSUER_URL")
API_BASE_URL = os.getenv("API_BASE_URL")
CHAT_API_PATH = os.getenv("CHAT_API_PATH")

In [83]:
assert CLIENT_ID is not None, "CLIENT_ID is missing"
assert CLIENT_SECRET is not None, "CLIENT_SECRET is missing"
assert TOKEN_ISSUER_URL is not None, "TOKEN_ISSUER_URL is missing"
assert API_BASE_URL is not None, "API_BASE_URL is missing"
assert CHAT_API_PATH is not None, "CHAT_API_PATH is missing"

print("API configuration loaded successfully.")

API configuration loaded successfully.


In [84]:
def get_access_token() -> str:
    token_data = {
        "grant_type": "client_credentials",
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
    }

    response = requests.post(
        TOKEN_ISSUER_URL,
        data=token_data,
        timeout=30
    )

    response.raise_for_status()

    return response.json()["access_token"]

In [85]:
def call_llm(messages: list[dict], temperature: float = 0.0) -> str:
    access_token = get_access_token()

    url = API_BASE_URL.rstrip("/") + CHAT_API_PATH

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
    }

    payload = {
        "messages": messages,
        "temperature": temperature,
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload,
        timeout=60
    )

    response.raise_for_status()

    result = response.json()

    return result["choices"][0]["message"]["content"]

In [86]:
test_messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant. Answer briefly."
    },
    {
        "role": "user",
        "content": "Return only the word: OK"
    }
]

test_response = call_llm(test_messages)

print(test_response)

OK


## 5. Prompt Design

In this section, we define the prompt for extracting structured constraints from the German municipal letter.

The prompt instructs the LLM to keep German source text unchanged and to return only valid JSON following the schema.

In [80]:
extraction_system_prompt = """
You are an information extraction system.

Your task is to extract structured construction-related constraints from a German municipal letter.

Rules:
- Return only valid JSON.
- Do not return markdown.
- Do not explain outside the JSON.
- Extract only information that is explicitly stated in the letter.
- Do not invent missing information.
- Keep the original German sentence in source_text.
- Use English field names.
- Use German values where they come directly from the original document.
"""

In [24]:
extraction_user_prompt = f"""
Extract all constraints from the following German municipal letter.

Return the result using this JSON structure:

{{
  "document_name": "Municipal_Letter.docx",
  "constraints": [
    {{
      "constraint_id": "C1",
      "source_text": "original German sentence",
      "constraint_type": "grabentiefe | arbeitszeit | grabensicherung | baumschutz | fortschrittsbericht",
      "parameter": "specific parameter being constrained",
      "operator": ">= | between | required | forbidden | allowed | optional",
      "value": "number, string, or null",
      "unit": "unit or null",
      "time_window": {{
        "start_time": "HH:MM or null",
        "end_time": "HH:MM or null"
      }},
      "condition": "additional condition or null",
      "obligation": "required | forbidden | allowed | optional",
      "confidence": 0.0
    }}
  ]
}}

German municipal letter:
{municipal_text}
"""

In [25]:
extraction_messages = [
    {
        "role": "system",
        "content": extraction_system_prompt
    },
    {
        "role": "user",
        "content": extraction_user_prompt
    }
]

## 6. Constraint Extraction

In this section, we call the LLM API and extract structured constraints from the German municipal letter.
The expected output is JSON that follows the extraction schema.

In [26]:
llm_extraction_response = call_llm(
    messages=extraction_messages,
    temperature=0.0
)

print(llm_extraction_response)

```json
{
  "document_name": "Municipal_Letter.docx",
  "constraints": [
    {
      "constraint_id": "C1",
      "source_text": "Die maximale Grabentiefe darf 80 cm nicht unterschreiten.",
      "constraint_type": "grabentiefe",
      "parameter": "Grabentiefe",
      "operator": ">=",
      "value": 80,
      "unit": "cm",
      "time_window": {
        "start_time": null,
        "end_time": null
      },
      "condition": null,
      "obligation": "required",
      "confidence": 1.0
    },
    {
      "constraint_id": "C2",
      "source_text": "Die Bauarbeiten sind ausschließlich in der Zeit von 7:00 bis 17:00 Uhr durchzuführen.",
      "constraint_type": "arbeitszeit",
      "parameter": "Bauarbeiten",
      "operator": "required",
      "value": null,
      "unit": null,
      "time_window": {
        "start_time": "07:00",
        "end_time": "17:00"
      },
      "condition": null,
      "obligation": "required",
      "confidence": 1.0
    },
    {
      "constraint_id": "C

In [28]:
import json

def clean_llm_json_response(response_text: str) -> str:
    """
    Removes markdown code fences that may be added by the LLM.

    Example:

    ```json
    {
        ...
    }
    ```

    becomes

    {
        ...
    }
    """

    cleaned = response_text.strip()

    if cleaned.startswith("```json"):
        cleaned = cleaned[len("```json"):].strip()

    if cleaned.startswith("```"):
        cleaned = cleaned[len("```"):].strip()

    if cleaned.endswith("```"):
        cleaned = cleaned[:-3].strip()

    return cleaned

cleaned_llm_response = clean_llm_json_response(
llm_extraction_response
)

extracted_constraints_dict = json.loads(
    cleaned_llm_response
)

extracted_constraints_dict

{'document_name': 'Municipal_Letter.docx',
 'constraints': [{'constraint_id': 'C1',
   'source_text': 'Die maximale Grabentiefe darf 80 cm nicht unterschreiten.',
   'constraint_type': 'grabentiefe',
   'parameter': 'Grabentiefe',
   'operator': '>=',
   'value': 80,
   'unit': 'cm',
   'time_window': {'start_time': None, 'end_time': None},
   'condition': None,
   'obligation': 'required',
   'confidence': 1.0},
  {'constraint_id': 'C2',
   'source_text': 'Die Bauarbeiten sind ausschließlich in der Zeit von 7:00 bis 17:00 Uhr durchzuführen.',
   'constraint_type': 'arbeitszeit',
   'parameter': 'Bauarbeiten',
   'operator': 'required',
   'value': None,
   'unit': None,
   'time_window': {'start_time': '07:00', 'end_time': '17:00'},
   'condition': None,
   'obligation': 'required',
   'confidence': 1.0},
  {'constraint_id': 'C3',
   'source_text': 'Offene Gräben sind über Nacht ordnungsgemäß abzudecken oder zu sichern.',
   'constraint_type': 'grabensicherung',
   'parameter': 'Offen

The municipal letter is unstructured text and therefore requires LLM-based extraction.

The ruleset already consists of explicit structured rules (R1-R5) and is therefore represented directly as structured Python objects.

This allows the LLM to focus on information extraction while keeping the compliance evaluation deterministic and explainable.

## 7. JSON Validation

In this section, we validate the extracted JSON against the Pydantic schema.

This ensures that the LLM output has the expected structure before it is used for rule evaluation.

In [29]:
validated_constraints = ExtractedConstraints.model_validate(
    extracted_constraints_dict
)

validated_constraints

ExtractedConstraints(document_name='Municipal_Letter.docx', constraints=[Constraint(constraint_id='C1', source_text='Die maximale Grabentiefe darf 80 cm nicht unterschreiten.', constraint_type='grabentiefe', parameter='Grabentiefe', operator='>=', value=80.0, unit='cm', time_window=TimeWindow(start_time=None, end_time=None), condition=None, obligation='required', confidence=1.0), Constraint(constraint_id='C2', source_text='Die Bauarbeiten sind ausschließlich in der Zeit von 7:00 bis 17:00 Uhr durchzuführen.', constraint_type='arbeitszeit', parameter='Bauarbeiten', operator='required', value=None, unit=None, time_window=TimeWindow(start_time='07:00', end_time='17:00'), condition=None, obligation='required', confidence=1.0), Constraint(constraint_id='C3', source_text='Offene Gräben sind über Nacht ordnungsgemäß abzudecken oder zu sichern.', constraint_type='grabensicherung', parameter='Offene Gräben', operator='required', value=None, unit=None, time_window=TimeWindow(start_time=None, end

In [30]:
print(f"Document name: {validated_constraints.document_name}")
print(f"Number of extracted constraints: {len(validated_constraints.constraints)}")

Document name: Municipal_Letter.docx
Number of extracted constraints: 5


In [31]:
for constraint in validated_constraints.constraints:
    print("Constraint ID:", constraint.constraint_id)
    print("Type:", constraint.constraint_type)
    print("Source:", constraint.source_text)
    print("Value:", constraint.value)
    print("Unit:", constraint.unit)
    print("-" * 50)

Constraint ID: C1
Type: grabentiefe
Source: Die maximale Grabentiefe darf 80 cm nicht unterschreiten.
Value: 80.0
Unit: cm
--------------------------------------------------
Constraint ID: C2
Type: arbeitszeit
Source: Die Bauarbeiten sind ausschließlich in der Zeit von 7:00 bis 17:00 Uhr durchzuführen.
Value: None
Unit: None
--------------------------------------------------
Constraint ID: C3
Type: grabensicherung
Source: Offene Gräben sind über Nacht ordnungsgemäß abzudecken oder zu sichern.
Value: None
Unit: None
--------------------------------------------------
Constraint ID: C4
Type: baumschutz
Source: Erdarbeiten im unmittelbaren Bereich von Bäumen sind nicht zulässig.
Value: None
Unit: None
--------------------------------------------------
Constraint ID: C5
Type: fortschrittsbericht
Source: Ein wöchentlicher Fortschrittsbericht ist der zuständigen Behörde vorzulegen.
Value: wöchentlich
Unit: None
--------------------------------------------------


## 8. Ruleset Structuring

In this section, we convert the German ruleset into structured Python data.

The rules are kept in German, but represented with English field names so they can be used by the Python rule engine.

In [32]:
rules = [
    {
        "rule_id": "R1",
        "rule_text": "Mindestgrabentiefe beträgt mindestens 60 cm",
        "rule_type": "grabentiefe",
        "parameter": "mindestgrabentiefe",
        "operator": ">=",
        "value": 60,
        "unit": "cm"
    },
    {
        "rule_id": "R2",
        "rule_text": "Bauarbeiten sind zwischen 6:00 und 20:00 Uhr zulässig",
        "rule_type": "arbeitszeit",
        "parameter": "bauarbeiten_zeitfenster",
        "operator": "between",
        "start_time": "6:00",
        "end_time": "20:00",
        "unit": "Uhr"
    },
    {
        "rule_id": "R3",
        "rule_text": "Offene Gräben müssen entweder gesichert oder abgedeckt werden",
        "rule_type": "grabensicherung",
        "parameter": "offene_gräben",
        "operator": "required",
        "allowed_actions": ["gesichert", "abgedeckt"]
    },
    {
        "rule_id": "R4",
        "rule_text": "Arbeiten innerhalb von 2 m um geschützte Bäume sind verboten",
        "rule_type": "baumschutz",
        "parameter": "geschützte_bäume",
        "operator": "forbidden",
        "distance_value": 2,
        "distance_unit": "m"
    },
    {
        "rule_id": "R5",
        "rule_text": "Wöchentliche Fortschrittsberichte sind zulässig, aber nicht verpflichtend",
        "rule_type": "fortschrittsbericht",
        "parameter": "wöchentlicher_fortschrittsbericht",
        "operator": "optional"
    }
]

In [33]:
for rule in rules:
    print(rule["rule_id"], "-", rule["rule_text"])

R1 - Mindestgrabentiefe beträgt mindestens 60 cm
R2 - Bauarbeiten sind zwischen 6:00 und 20:00 Uhr zulässig
R3 - Offene Gräben müssen entweder gesichert oder abgedeckt werden
R4 - Arbeiten innerhalb von 2 m um geschützte Bäume sind verboten
R5 - Wöchentliche Fortschrittsberichte sind zulässig, aber nicht verpflichtend


## 9. Rule Evaluation Strategy

In this section, we define how extracted constraints are evaluated against the simplified ruleset.

Each rule is mapped to a specific comparison strategy.

The goal is to identify compliance, partial compliance, violations, or cases that require manual review, with explicit reasoning.

In [34]:
evaluation_strategy = {
    "R1": {
        "comparison_type": "numeric_minimum_check",
        "rule_focus": "mindestgrabentiefe",
        "logic": "extracted_depth >= required_minimum_depth",
        "possible_status": [
            "compliant",
            "violation"
        ]
    },

    "R2": {
        "comparison_type": "time_window_check",
        "rule_focus": "bauarbeiten_zeitfenster",
        "logic": "municipal_time_window inside allowed_ruleset_time_window",
        "possible_status": [
            "compliant",
            "violation"
        ]
    },

    "R3": {
        "comparison_type": "obligation_check",
        "rule_focus": "offene_gräben",
        "logic": "letter requires either gesichert or abgedeckt",
        "possible_status": [
            "compliant",
            "violation"
        ]
    },

    "R4": {
        "comparison_type": "ambiguity_check",
        "rule_focus": "baumschutz",
        "logic": "letter forbids work near trees but does not specify exact 2 m distance or protected trees",
        "possible_status": [
            "compliant",
            "partial_compliance",
            "needs_review",
            "violation"
        ]
    },

    "R5": {
        "comparison_type": "additional_requirement_check",
        "rule_focus": "fortschrittsbericht",
        "logic": "ruleset says reports are allowed but not mandatory; letter requires weekly report",
        "possible_status": [
            "not_applicable",
            "compliant"
        ]
    }
}

In [35]:
for rule_id, strategy in evaluation_strategy.items():
    print(f"{rule_id}")
    print(f"Comparison Type: {strategy['comparison_type']}")
    print(f"Logic: {strategy['logic']}")
    print(f"Possible Status: {strategy['possible_status']}")
    print("-" * 60)

R1
Comparison Type: numeric_minimum_check
Logic: extracted_depth >= required_minimum_depth
Possible Status: ['compliant', 'violation']
------------------------------------------------------------
R2
Comparison Type: time_window_check
Logic: municipal_time_window inside allowed_ruleset_time_window
Possible Status: ['compliant', 'violation']
------------------------------------------------------------
R3
Comparison Type: obligation_check
Logic: letter requires either gesichert or abgedeckt
Possible Status: ['compliant', 'violation']
------------------------------------------------------------
R4
Comparison Type: ambiguity_check
Logic: letter forbids work near trees but does not specify exact 2 m distance or protected trees
Possible Status: ['compliant', 'partial_compliance', 'needs_review', 'violation']
------------------------------------------------------------
R5
Comparison Type: additional_requirement_check
Logic: ruleset says reports are allowed but not mandatory; letter requires we

### 9.1 Rule R1: Mindestgrabentiefe

R1 checks whether the trench depth required in the municipal letter satisfies the minimum depth required by the ruleset.

Ruleset:
R1: Mindestgrabentiefe beträgt mindestens 60 cm

The municipal letter states:
Die maximale Grabentiefe darf 80 cm nicht unterschreiten.

In [37]:
def find_constraint_by_type(validated_constraints, constraint_type: str):
    for constraint in validated_constraints.constraints:
        if constraint.constraint_type == constraint_type:
            return constraint
    return None

In [38]:
def evaluate_r1_trench_depth(validated_constraints, rules):
    rule = next(rule for rule in rules if rule["rule_id"] == "R1")
    constraint = find_constraint_by_type(
        validated_constraints,
        "grabentiefe"
    )

    if constraint is None:
        return ComplianceResult(
            rule_id=rule["rule_id"],
            rule_text=rule["rule_text"],
            related_constraint_id=None,
            status="needs_review",
            evidence_from_letter=None,
            reasoning="No trench depth constraint was extracted from the municipal letter.",
            recommendation="Review the municipal letter manually."
        )

    extracted_depth = float(constraint.value)
    required_minimum_depth = float(rule["value"])

    if extracted_depth >= required_minimum_depth:
        status = "compliant"
        reasoning = (
            f"The municipal letter requires a trench depth of at least "
            f"{extracted_depth:g} {constraint.unit}. "
            f"The ruleset requires at least {required_minimum_depth:g} {rule['unit']}. "
            f"Because {extracted_depth:g} >= {required_minimum_depth:g}, the rule is compliant."
        )
        recommendation = None
    else:
        status = "violation"
        reasoning = (
            f"The municipal letter requires a trench depth of "
            f"{extracted_depth:g} {constraint.unit}. "
            f"The ruleset requires at least {required_minimum_depth:g} {rule['unit']}. "
            f"Because {extracted_depth:g} < {required_minimum_depth:g}, the rule is violated."
        )
        recommendation = "Increase the required trench depth to meet the minimum rule."

    return ComplianceResult(
        rule_id=rule["rule_id"],
        rule_text=rule["rule_text"],
        related_constraint_id=constraint.constraint_id,
        status=status,
        evidence_from_letter=constraint.source_text,
        reasoning=reasoning,
        recommendation=recommendation
    )

In [39]:
r1_result = evaluate_r1_trench_depth(
    validated_constraints,
    rules
)

r1_result

ComplianceResult(rule_id='R1', rule_text='Mindestgrabentiefe beträgt mindestens 60 cm', related_constraint_id='C1', status='compliant', evidence_from_letter='Die maximale Grabentiefe darf 80 cm nicht unterschreiten.', reasoning='The municipal letter requires a trench depth of at least 80 cm. The ruleset requires at least 60 cm. Because 80 >= 60, the rule is compliant.', recommendation=None)

### 9.2 Rule R2: Bauarbeiten Zeitfenster

R2 checks whether the construction working hours from the municipal letter are within the allowed time window from the ruleset.

Ruleset:
R2: Bauarbeiten sind zwischen 6:00 und 20:00 Uhr zulässig

The municipal letter states:
Die Bauarbeiten sind ausschließlich in der Zeit von 7:00 bis 17:00 Uhr durchzuführen.

In [40]:
from datetime import datetime


def parse_time_to_minutes(time_text: str) -> int:
    """
    Converts a time string like '7:00' into minutes after midnight.
    """
    parsed_time = datetime.strptime(time_text, "%H:%M")
    return parsed_time.hour * 60 + parsed_time.minute

In [41]:
def evaluate_r2_working_hours(validated_constraints, rules):
    rule = next(rule for rule in rules if rule["rule_id"] == "R2")

    constraint = find_constraint_by_type(
        validated_constraints,
        "arbeitszeit"
    )

    if constraint is None or constraint.time_window is None:
        return ComplianceResult(
            rule_id=rule["rule_id"],
            rule_text=rule["rule_text"],
            related_constraint_id=None,
            status="needs_review",
            evidence_from_letter=None,
            reasoning="No working-hours constraint was extracted from the municipal letter.",
            recommendation="Review the municipal letter manually."
        )

    municipal_start = parse_time_to_minutes(
        constraint.time_window.start_time
    )
    municipal_end = parse_time_to_minutes(
        constraint.time_window.end_time
    )

    rule_start = parse_time_to_minutes(
        rule["start_time"]
    )
    rule_end = parse_time_to_minutes(
        rule["end_time"]
    )

    is_inside_allowed_window = (
        municipal_start >= rule_start
        and municipal_end <= rule_end
    )

    if is_inside_allowed_window:
        status = "compliant"
        reasoning = (
            f"The municipal letter allows Bauarbeiten only from "
            f"{constraint.time_window.start_time} to {constraint.time_window.end_time}. "
            f"The ruleset allows Bauarbeiten between {rule['start_time']} and {rule['end_time']}. "
            f"Because the municipal time window is inside the allowed ruleset window, the rule is compliant."
        )
        recommendation = None
    else:
        status = "violation"
        reasoning = (
            f"The municipal letter allows Bauarbeiten from "
            f"{constraint.time_window.start_time} to {constraint.time_window.end_time}. "
            f"The ruleset allows Bauarbeiten only between {rule['start_time']} and {rule['end_time']}. "
            f"Because the municipal time window is outside the allowed ruleset window, the rule is violated."
        )
        recommendation = "Adjust the working hours to stay within the allowed ruleset time window."

    return ComplianceResult(
        rule_id=rule["rule_id"],
        rule_text=rule["rule_text"],
        related_constraint_id=constraint.constraint_id,
        status=status,
        evidence_from_letter=constraint.source_text,
        reasoning=reasoning,
        recommendation=recommendation
    )

In [42]:
r2_result = evaluate_r2_working_hours(
    validated_constraints,
    rules
)

r2_result

ComplianceResult(rule_id='R2', rule_text='Bauarbeiten sind zwischen 6:00 und 20:00 Uhr zulässig', related_constraint_id='C2', status='compliant', evidence_from_letter='Die Bauarbeiten sind ausschließlich in der Zeit von 7:00 bis 17:00 Uhr durchzuführen.', reasoning='The municipal letter allows Bauarbeiten only from 07:00 to 17:00. The ruleset allows Bauarbeiten between 6:00 and 20:00. Because the municipal time window is inside the allowed ruleset window, the rule is compliant.', recommendation=None)

### 9.3 Rule R3: Offene Gräben sichern oder abdecken

R3 checks whether open trenches are required to be secured or covered.

Ruleset:
R3: Offene Gräben müssen entweder gesichert oder abgedeckt werden

The municipal letter states:
Offene Gräben sind über Nacht ordnungsgemäß abzudecken oder zu sichern.

In [87]:
def evaluate_r3_trench_security(validated_constraints, rules):
    rule = next(rule for rule in rules if rule["rule_id"] == "R3")

    constraint = find_constraint_by_type(
        validated_constraints,
        "grabensicherung"
    )

    if constraint is None:
        return ComplianceResult(
            rule_id=rule["rule_id"],
            rule_text=rule["rule_text"],
            related_constraint_id=None,
            status="needs_review",
            evidence_from_letter=None,
            reasoning="No trench security constraint was extracted from the municipal letter.",
            recommendation="Review the municipal letter manually."
        )

    source_text_lower = constraint.source_text.lower()
    condition_lower = (constraint.condition or "").lower()

    combined_text = f"{source_text_lower} {condition_lower}"

    requires_cover_or_secure = (
        constraint.obligation == "required"
        and (
            "abzudecken" in combined_text
            or "abgedeckt" in combined_text
            or "sichern" in combined_text
            or "gesichert" in combined_text
        )
    )

    mentions_limited_time_condition = (
        "nacht" in combined_text
        or "nachts" in combined_text
        or "wochenende" in combined_text
        or "wochenenden" in combined_text
        or "feiertag" in combined_text
        or "feiertage" in combined_text
        or "außerhalb der arbeitszeit" in combined_text
        or "außerhalb der arbeitszeiten" in combined_text
    )

    if requires_cover_or_secure and mentions_limited_time_condition:
        status = "partial_compliance"
        reasoning = (
            f"The municipal letter contains a trench-security requirement: "
            f"'{constraint.source_text}'. "
            f"This is aligned with the ruleset because open trenches must be covered or secured. "
            f"However, the municipal letter links this requirement to a specific time condition. "
            f"The ruleset does not explicitly contain such a limitation. "
            f"Therefore, the rule is treated as partial compliance."
        )
        recommendation = (
            "Clarify whether open trenches must be secured or covered generally, "
            "or only during the specified time condition."
        )

    elif requires_cover_or_secure:
        status = "compliant"
        reasoning = (
            f"The municipal letter contains a trench-security requirement: "
            f"'{constraint.source_text}'. "
            f"The ruleset requires open trenches to be secured or covered. "
            f"Because the municipal letter contains this requirement without a limiting condition, "
            f"the rule is compliant."
        )
        recommendation = None

    else:
        status = "violation"
        reasoning = (
            f"The ruleset requires open trenches to be secured or covered. "
            f"The extracted municipal constraint does not clearly contain this requirement."
        )
        recommendation = "Add an explicit requirement to secure or cover open trenches."

    return ComplianceResult(
        rule_id=rule["rule_id"],
        rule_text=rule["rule_text"],
        related_constraint_id=constraint.constraint_id,
        status=status,
        evidence_from_letter=constraint.source_text,
        reasoning=reasoning,
        recommendation=recommendation
    )

In [88]:
r3_result = evaluate_r3_trench_security(
    validated_constraints,
    rules
)

r3_result

ComplianceResult(rule_id='R3', rule_text='Offene Gräben müssen entweder gesichert oder abgedeckt werden', related_constraint_id='C3', status='partial_compliance', evidence_from_letter='Offene Gräben sind über Nacht ordnungsgemäß abzudecken oder zu sichern.', reasoning="The municipal letter contains a trench-security requirement: 'Offene Gräben sind über Nacht ordnungsgemäß abzudecken oder zu sichern.'. This is aligned with the ruleset because open trenches must be covered or secured. However, the municipal letter links this requirement to a specific time condition. The ruleset does not explicitly contain such a limitation. Therefore, the rule is treated as partial compliance.", recommendation='Clarify whether open trenches must be secured or covered generally, or only during the specified time condition.')

## 9.4 Rule R4: Baumschutz

R4 checks whether work around protected trees is forbidden.

Ruleset:
R4: Arbeiten innerhalb von 2 m um geschützte Bäume sind verboten

The municipal letter states:
Erdarbeiten im unmittelbaren Bereich von Bäumen sind nicht zulässig.

In [113]:
def evaluate_r4_tree_protection(validated_constraints, rules):
    rule = next(rule for rule in rules if rule["rule_id"] == "R4")

    constraint = find_constraint_by_type(
        validated_constraints,
        "baumschutz"
    )

    if constraint is None:
        return ComplianceResult(
            rule_id=rule["rule_id"],
            rule_text=rule["rule_text"],
            related_constraint_id=None,
            status="needs_review",
            evidence_from_letter=None,
            reasoning="No tree protection constraint was extracted from the municipal letter.",
            recommendation="Review the municipal letter manually."
        )

    source_text_lower = constraint.source_text.lower()

    mentions_trees = (
        "baum" in source_text_lower
        or "bäumen" in source_text_lower
        or "bäume" in source_text_lower
    )

    forbids_work = (
        "nicht zulässig" in source_text_lower
        or "verboten" in source_text_lower
        or constraint.obligation == "forbidden"
    )

    mentions_exact_distance = (
        "2 m" in source_text_lower
        or "2m" in source_text_lower
        or "zwei meter" in source_text_lower
    )

    mentions_protected_trees = (
        "geschützte" in source_text_lower
        or "geschützten" in source_text_lower
    )

    if mentions_trees and forbids_work and mentions_exact_distance and mentions_protected_trees:
        status = "compliant"
        reasoning = (
            "The municipal letter forbids work within the exact 2 m area around protected trees. "
            "This directly matches the ruleset requirement."
        )
        recommendation = None

    elif mentions_trees and forbids_work:
        status = "partial_compliance"
        reasoning = (
            "The municipal letter forbids Erdarbeiten im unmittelbaren Bereich von Bäumen. "
            "This is directionally aligned with the ruleset because work near trees is restricted. "
            "However, the letter does not explicitly mention the exact 2 m distance or the term "
            "'geschützte Bäume'. Therefore, the rule is treated as partial compliance."
        )
        recommendation = (
            "Clarify whether 'im unmittelbaren Bereich von Bäumen' corresponds to "
            "the required 2 m protection zone around protected trees."
        )

    else:
        status = "violation"
        reasoning = (
            "The ruleset forbids work within 2 m around protected trees, "
            "but the municipal letter does not clearly contain such a restriction."
        )
        recommendation = (
            "Add an explicit restriction for work within 2 m around protected trees."
        )

    return ComplianceResult(
        rule_id=rule["rule_id"],
        rule_text=rule["rule_text"],
        related_constraint_id=constraint.constraint_id,
        status=status,
        evidence_from_letter=constraint.source_text,
        reasoning=reasoning,
        recommendation=recommendation
    )

In [114]:
r4_result = evaluate_r4_tree_protection(
    validated_constraints,
    rules
)

r4_result

ComplianceResult(rule_id='R4', rule_text='Arbeiten innerhalb von 2 m um geschützte Bäume sind verboten', related_constraint_id='C4', status='partial_compliance', evidence_from_letter='Erdarbeiten im unmittelbaren Bereich von Bäumen sind nicht zulässig.', reasoning="The municipal letter forbids Erdarbeiten im unmittelbaren Bereich von Bäumen. This is directionally aligned with the ruleset because work near trees is restricted. However, the letter does not explicitly mention the exact 2 m distance or the term 'geschützte Bäume'. Therefore, the rule is treated as partial compliance.", recommendation="Clarify whether 'im unmittelbaren Bereich von Bäumen' corresponds to the required 2 m protection zone around protected trees.")

### 9.5 Rule R5: Wöchentliche Fortschrittsberichte

R5 checks whether weekly progress reports are handled according to the ruleset.

Ruleset:
R5: Wöchentliche Fortschrittsberichte sind zulässig, aber nicht verpflichtend

The municipal letter states:
Ein wöchentlicher Fortschrittsbericht ist der zuständigen Behörde vorzulegen.

In [117]:
def evaluate_r5_progress_report(validated_constraints, rules):
    rule = next(rule for rule in rules if rule["rule_id"] == "R5")

    constraint = find_constraint_by_type(
        validated_constraints,
        "fortschrittsbericht"
    )

    if constraint is None:
        return ComplianceResult(
            rule_id=rule["rule_id"],
            rule_text=rule["rule_text"],
            related_constraint_id=None,
            status="not_applicable",
            evidence_from_letter=None,
            reasoning=(
                "No progress-report constraint was extracted from the municipal letter. "
                "The ruleset allows weekly progress reports but does not require them. "
                "Therefore, no compliance issue is identified for this rule."
            ),
            recommendation=None
        )

    return ComplianceResult(
        rule_id=rule["rule_id"],
        rule_text=rule["rule_text"],
        related_constraint_id=constraint.constraint_id,
        status="compliant",
        evidence_from_letter=constraint.source_text,
        reasoning=(
            f"The municipal letter contains a progress-report constraint: "
            f"'{constraint.source_text}'. "
            f"The ruleset states that weekly progress reports are allowed but not mandatory. "
            f"Because the municipal letter includes a progress-report requirement that is allowed by the ruleset, "
            f"the rule is compliant."
        ),
        recommendation=None
    )

In [118]:
r5_result = evaluate_r5_progress_report(
    validated_constraints,
    rules
)

r5_result

ComplianceResult(rule_id='R5', rule_text='Wöchentliche Fortschrittsberichte sind zulässig, aber nicht verpflichtend', related_constraint_id='C5', status='compliant', evidence_from_letter='Ein wöchentlicher Fortschrittsbericht ist der zuständigen Behörde vorzulegen.', reasoning="The municipal letter contains a progress-report constraint: 'Ein wöchentlicher Fortschrittsbericht ist der zuständigen Behörde vorzulegen.'. The ruleset states that weekly progress reports are allowed but not mandatory. Because the municipal letter includes a progress-report requirement that is allowed by the ruleset, the rule is compliant.", recommendation=None)

## 10. Complete Rule Engine

In this section, all rule evaluations are executed and combined into a single rule engine result.

The goal is to evaluate all extracted constraints against the simplified ruleset and return structured compliance results.

In [94]:
all_rule_results = [
    r1_result,
    r2_result,
    r3_result,
    r4_result,
    r5_result
]

all_rule_results

[ComplianceResult(rule_id='R1', rule_text='Mindestgrabentiefe beträgt mindestens 60 cm', related_constraint_id='C1', status='compliant', evidence_from_letter='Die maximale Grabentiefe darf 80 cm nicht unterschreiten.', reasoning='The municipal letter requires a trench depth of at least 80 cm. The ruleset requires at least 60 cm. Because 80 >= 60, the rule is compliant.', recommendation=None),
 ComplianceResult(rule_id='R2', rule_text='Bauarbeiten sind zwischen 6:00 und 20:00 Uhr zulässig', related_constraint_id='C2', status='compliant', evidence_from_letter='Die Bauarbeiten sind ausschließlich in der Zeit von 7:00 bis 17:00 Uhr durchzuführen.', reasoning='The municipal letter allows Bauarbeiten only from 07:00 to 17:00. The ruleset allows Bauarbeiten between 6:00 and 20:00. Because the municipal time window is inside the allowed ruleset window, the rule is compliant.', recommendation=None),
 ComplianceResult(rule_id='R3', rule_text='Offene Gräben müssen entweder gesichert oder abgedeck

In [95]:
print(f"Number of evaluated rules: {len(all_rule_results)}")

Number of evaluated rules: 5


In [96]:
for result in all_rule_results:
    print(f"Rule ID: {result.rule_id}")
    print(f"Status: {result.status}")
    print(f"Evidence: {result.evidence_from_letter}")
    print(f"Reasoning: {result.reasoning}")
    print("-" * 80)

Rule ID: R1
Status: compliant
Evidence: Die maximale Grabentiefe darf 80 cm nicht unterschreiten.
Reasoning: The municipal letter requires a trench depth of at least 80 cm. The ruleset requires at least 60 cm. Because 80 >= 60, the rule is compliant.
--------------------------------------------------------------------------------
Rule ID: R2
Status: compliant
Evidence: Die Bauarbeiten sind ausschließlich in der Zeit von 7:00 bis 17:00 Uhr durchzuführen.
Reasoning: The municipal letter allows Bauarbeiten only from 07:00 to 17:00. The ruleset allows Bauarbeiten between 6:00 and 20:00. Because the municipal time window is inside the allowed ruleset window, the rule is compliant.
--------------------------------------------------------------------------------
Rule ID: R3
Status: partial_compliance
Evidence: Offene Gräben sind über Nacht ordnungsgemäß abzudecken oder zu sichern.
Reasoning: The municipal letter contains a trench-security requirement: 'Offene Gräben sind über Nacht ordnungsge

## 11. Final Compliance Assessment JSON

In this section, we generate the final machine-readable compliance assessment.

The assessment contains:

- the document name
- the overall compliance status
- all rule-level results
- reasoning
- evidence from the German municipal letter

In [97]:
def determine_overall_status(rule_results):
    statuses = [
        result.status
        for result in rule_results
    ]

    if "violation" in statuses:
        return "violation"

    if "needs_review" in statuses:
        return "needs_review"

    if "partial_compliance" in statuses:
        return "partial_compliance"

    return "compliant"

In [98]:
overall_status = determine_overall_status(
    all_rule_results
)

print("Overall status:", overall_status)

Overall status: partial_compliance


In [99]:
final_compliance_assessment = ComplianceAssessment(
    document_name=validated_constraints.document_name,
    overall_status=overall_status,
    results=all_rule_results
)

final_compliance_assessment

ComplianceAssessment(document_name='Municipal_Letter.docx', overall_status='partial_compliance', results=[ComplianceResult(rule_id='R1', rule_text='Mindestgrabentiefe beträgt mindestens 60 cm', related_constraint_id='C1', status='compliant', evidence_from_letter='Die maximale Grabentiefe darf 80 cm nicht unterschreiten.', reasoning='The municipal letter requires a trench depth of at least 80 cm. The ruleset requires at least 60 cm. Because 80 >= 60, the rule is compliant.', recommendation=None), ComplianceResult(rule_id='R2', rule_text='Bauarbeiten sind zwischen 6:00 und 20:00 Uhr zulässig', related_constraint_id='C2', status='compliant', evidence_from_letter='Die Bauarbeiten sind ausschließlich in der Zeit von 7:00 bis 17:00 Uhr durchzuführen.', reasoning='The municipal letter allows Bauarbeiten only from 07:00 to 17:00. The ruleset allows Bauarbeiten between 6:00 and 20:00. Because the municipal time window is inside the allowed ruleset window, the rule is compliant.', recommendation

In [100]:
final_compliance_assessment_json = final_compliance_assessment.model_dump()

final_compliance_assessment_json

{'document_name': 'Municipal_Letter.docx',
 'overall_status': 'partial_compliance',
 'results': [{'rule_id': 'R1',
   'rule_text': 'Mindestgrabentiefe beträgt mindestens 60 cm',
   'related_constraint_id': 'C1',
   'status': 'compliant',
   'evidence_from_letter': 'Die maximale Grabentiefe darf 80 cm nicht unterschreiten.',
   'reasoning': 'The municipal letter requires a trench depth of at least 80 cm. The ruleset requires at least 60 cm. Because 80 >= 60, the rule is compliant.',
   'recommendation': None},
  {'rule_id': 'R2',
   'rule_text': 'Bauarbeiten sind zwischen 6:00 und 20:00 Uhr zulässig',
   'related_constraint_id': 'C2',
   'status': 'compliant',
   'evidence_from_letter': 'Die Bauarbeiten sind ausschließlich in der Zeit von 7:00 bis 17:00 Uhr durchzuführen.',
   'reasoning': 'The municipal letter allows Bauarbeiten only from 07:00 to 17:00. The ruleset allows Bauarbeiten between 6:00 and 20:00. Because the municipal time window is inside the allowed ruleset window, the ru

## 12. Reasoning and Evidence Verification

In this section, we verify that every rule result contains explicit reasoning and evidence from the German municipal letter.

This ensures that the compliance assessment is explainable and traceable.

In [101]:
for result in final_compliance_assessment.results:
    assert result.reasoning is not None
    assert result.reasoning.strip() != ""

    assert result.evidence_from_letter is not None
    assert result.evidence_from_letter.strip() != ""

print("All rule results contain reasoning and German evidence.")

All rule results contain reasoning and German evidence.


In [102]:
for result in final_compliance_assessment.results:
    print(f"Rule ID: {result.rule_id}")
    print(f"Status: {result.status}")
    print(f"German Evidence: {result.evidence_from_letter}")
    print(f"Reasoning: {result.reasoning}")
    print("-" * 80)

Rule ID: R1
Status: compliant
German Evidence: Die maximale Grabentiefe darf 80 cm nicht unterschreiten.
Reasoning: The municipal letter requires a trench depth of at least 80 cm. The ruleset requires at least 60 cm. Because 80 >= 60, the rule is compliant.
--------------------------------------------------------------------------------
Rule ID: R2
Status: compliant
German Evidence: Die Bauarbeiten sind ausschließlich in der Zeit von 7:00 bis 17:00 Uhr durchzuführen.
Reasoning: The municipal letter allows Bauarbeiten only from 07:00 to 17:00. The ruleset allows Bauarbeiten between 6:00 and 20:00. Because the municipal time window is inside the allowed ruleset window, the rule is compliant.
--------------------------------------------------------------------------------
Rule ID: R3
Status: partial_compliance
German Evidence: Offene Gräben sind über Nacht ordnungsgemäß abzudecken oder zu sichern.
Reasoning: The municipal letter contains a trench-security requirement: 'Offene Gräben sind 

## 13. Save JSON Outputs

In this section, we save the extracted constraints and the final compliance assessment as JSON files.

These files are machine-readable outputs that can be used by downstream automation or agent-based workflows.

In [103]:
from pathlib import Path
import json

output_dir = Path("../outputs")
output_dir.mkdir(exist_ok=True)

In [104]:
extracted_constraints_path = output_dir / "extracted_constraints.json"

with open(extracted_constraints_path, "w", encoding="utf-8") as file:
    json.dump(
        validated_constraints.model_dump(),
        file,
        ensure_ascii=False,
        indent=2
    )

print(f"Saved extracted constraints to: {extracted_constraints_path}")

Saved extracted constraints to: ..\outputs\extracted_constraints.json


In [105]:
validation_result_path = output_dir / "validation_result.json"

with open(validation_result_path, "w", encoding="utf-8") as file:
    json.dump(
        final_compliance_assessment.model_dump(),
        file,
        ensure_ascii=False,
        indent=2
    )

print(f"Saved validation result to: {validation_result_path}")

Saved validation result to: ..\outputs\validation_result.json


In [109]:
assert extracted_constraints_path.exists()
assert validation_result_path.exists()

print("Both JSON output files were saved successfully.")

Both JSON output files were saved successfully.


## 14. End-to-End Workflow Verification

In this section, we verify that the complete workflow executes successfully from document ingestion to final JSON output.

The goal is to ensure that all components of the prototype work together correctly.

In [119]:
workflow_checklist = {
    "documents_loaded": bool(municipal_text and ruleset_text),
    "constraints_extracted": validated_constraints is not None,
    "rules_parsed": len(rules) > 0,
    "rule_engine_executed": len(all_rule_results) == 5,
    "compliance_assessment_created": final_compliance_assessment is not None,
    "constraints_json_saved": extracted_constraints_path.exists(),
    "assessment_json_saved": validation_result_path.exists()
}

workflow_checklist

{'documents_loaded': True,
 'constraints_extracted': True,
 'rules_parsed': True,
 'rule_engine_executed': True,
 'compliance_assessment_created': True,
 'constraints_json_saved': True,
 'assessment_json_saved': True}

In [120]:
for step_name, status in workflow_checklist.items():
    print(f"{step_name}: {'PASS' if status else 'FAIL'}")

documents_loaded: PASS
constraints_extracted: PASS
rules_parsed: PASS
rule_engine_executed: PASS
compliance_assessment_created: PASS
constraints_json_saved: PASS
assessment_json_saved: PASS


In [122]:
assert all(workflow_checklist.values())

print("End-to-end workflow completed successfully.")

End-to-end workflow completed successfully.
